# 04 — Cross-Match Diagnostics

**Plots:** match counts per image · Gaia magnitude completeness · sky distribution · Gaia vs HST magnitude · transformation parameter summary

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = '..'
FIELD_NAME = 'Sculptor_dSph'
TELESCOPE  = 'HST'
IM_TYPE    = '_flc'
# ─────────────────────────────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path

from bp3m.pipeline.explore_utils import (
    load_gaia_catalog, load_cross_match_results, load_bp3m_results,
)

DATA = Path(OUTPUT_DIR)
_gaia_files = sorted((DATA / FIELD_NAME / 'Gaia').glob('*_gaia.csv'))
gaia = load_gaia_catalog(_gaia_files[0])

try:
    xm = load_cross_match_results(DATA, FIELD_NAME, TELESCOPE, IM_TYPE)
    print(f'Cross-matched images: {xm["image_name"].nunique()}')
    print(f'Total matched detections: {len(xm)}')
    print(f'Unique Gaia stars matched: {xm["gaia_source_id"].nunique()}')
except FileNotFoundError as e:
    xm = None
    print(f'Cross-match data not found: {e}')

# Transformation parameters come from BP3M_results/image_transformations.csv
try:
    bp3m = load_bp3m_results(DATA / FIELD_NAME / 'BP3M_results')
    bp3m_images = bp3m.get('images')   # image_transformations.csv
    if bp3m_images is not None:
        print(f'BP3M images: {len(bp3m_images)}')
except FileNotFoundError:
    bp3m_images = None
    print('BP3M results not found.')

In [ ]:
# ── Match counts per image ────────────────────────────────────────────────────
if xm is not None:
    counts = xm.groupby('image_name').size().sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(max(6, len(counts) * 0.4), 5))
    ax.bar(range(len(counts)), counts.values, color='steelblue', edgecolor='white', lw=0.5)
    ax.set_xticks(range(len(counts)))
    ax.set_xticklabels(counts.index, rotation=90, fontsize=7)
    ax.set_ylabel('N matched stars')
    ax.set_title(f'{FIELD_NAME} — Matched stars per image')
    ax.axhline(counts.median(), color='darkorange', lw=1.5, ls='--',
               label=f'Median = {counts.median():.0f}')
    ax.legend()
    plt.tight_layout()
    plt.savefig(DATA / FIELD_NAME / 'plots_04_match_counts.png', dpi=150)
    plt.show()
else:
    print('Cross-match data required for this plot.')

In [ ]:
# ── Gaia G magnitude completeness ────────────────────────────────────────────
if xm is not None:
    gid_col = 'gaia_source_id' if 'gaia_source_id' in xm.columns else None
    matched_ids = set(xm[gid_col].unique()) if gid_col else set()
    gaia_matched = gaia['source_id'].isin(matched_ids) if 'source_id' in gaia.columns else \
                   gaia.index.isin([])

    bins = np.arange(gaia['gmag'].min(), gaia['gmag'].max() + 0.5, 0.5)

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(gaia['gmag'], bins=bins, color='steelblue', alpha=0.7,
            label='All Gaia stars', edgecolor='white', lw=0.5)
    ax.hist(gaia.loc[gaia_matched, 'gmag'], bins=bins,
            color='darkorange', alpha=0.7,
            label=f'Matched ({gaia_matched.sum()})', edgecolor='white', lw=0.5)
    ax.set_xlabel('Gaia G (mag)')
    ax.set_ylabel('N stars')
    ax.legend()
    ax.set_title(f'{FIELD_NAME} — Gaia completeness of cross-match')
    plt.tight_layout()
    plt.savefig(DATA / FIELD_NAME / 'plots_04_completeness.png', dpi=150)
    plt.show()
else:
    print('Cross-match data required for this plot.')

In [ ]:
# ── Sky distribution of matched stars (colour by image) ───────────────────────
if xm is not None:
    images = sorted(xm['image_name'].unique())
    img_to_idx = {img: i for i, img in enumerate(images)}
    color_vals = xm['image_name'].map(img_to_idx).values

    fig, ax = plt.subplots(figsize=(8, 7))
    ra_col  = next((c for c in ['gaia_ra_prop', 'ra']  if c in xm.columns), None)
    dec_col = next((c for c in ['gaia_dec_prop', 'dec'] if c in xm.columns), None)

    if ra_col and dec_col:
        sc = ax.scatter(xm[ra_col], xm[dec_col],
                        c=color_vals, cmap='tab20',
                        vmin=0, vmax=max(len(images) - 1, 1),
                        s=3, alpha=0.5, rasterized=True)
        cbar = plt.colorbar(sc, ax=ax)
        cbar.set_label('Image index')
        cbar.set_ticks(range(len(images)))
        cbar.set_ticklabels([f'{i}: {img}' for i, img in enumerate(images)],
                            fontsize=max(4, 7 - len(images) // 8))
        ax.set_xlabel('R.A. (deg)')
        ax.set_ylabel('Dec. (deg)')
        ax.set_title(f'{FIELD_NAME} — Matched stars by image ({len(images)} images)')
        plt.tight_layout()
        plt.savefig(DATA / FIELD_NAME / 'plots_04_sky_by_image.png', dpi=150)
        plt.show()
    else:
        print(f'No ra/dec columns found. Available: {list(xm.columns)}')
else:
    print('Cross-match data required for this plot.')

In [ ]:
# ── Gaia G vs HST instrumental magnitude ─────────────────────────────────────
if xm is not None:
    hst_mag_col = next((c for c in ['hst_mag_st_gdc', 'hst_mag_gdc', 'hst_mag', 'mag_hst']
                        if c in xm.columns), None)
    gmag_col    = next((c for c in ['gaia_gmag', 'gmag'] if c in xm.columns), None)

    if hst_mag_col and gmag_col:
        ok = np.isfinite(xm[gmag_col].values) & np.isfinite(xm[hst_mag_col].values)
        g_ok = xm.loc[ok, gmag_col].values
        h_ok = xm.loc[ok, hst_mag_col].values

        # Running median
        sort_idx = np.argsort(g_ok)
        g_s, h_s = g_ok[sort_idx], h_ok[sort_idx]
        window = max(1, len(g_s) // 50)
        g_med = np.convolve(g_s, np.ones(window)/window, mode='valid')
        h_med = np.convolve(h_s, np.ones(window)/window, mode='valid')

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(g_ok, h_ok, s=3, alpha=0.3, c='steelblue', rasterized=True)
        ax.plot(g_med, h_med, color='darkorange', lw=2, label='Running median')
        ax.set_xlabel('Gaia G (mag)')
        ax.set_ylabel(f'{hst_mag_col}')
        ax.set_title(f'{FIELD_NAME} — Gaia G vs HST magnitude')
        ax.legend()
        plt.tight_layout()
        plt.savefig(DATA / FIELD_NAME / 'plots_04_gaia_vs_hst_mag.png', dpi=150)
        plt.show()
    else:
        print(f'Need Gaia gmag and HST mag column. Found: {list(xm.columns)}')
else:
    print('Cross-match data required for this plot.')

In [ ]:
# ── BP3M transformation parameter summary ────────────────────────────────────
# These come from BP3M_results/image_transformations.csv, not the per-image
# cross-match transformation.csv (which only has the rough initial solution).
if bp3m_images is not None:
    param_cols = ['pixel_scale_mas', 'rotation_deg', 'on_skew', 'off_skew',
                  'delta_ra0_mas', 'delta_dec0_mas', 'alpha']
    present = [c for c in param_cols if c in bp3m_images.columns]

    n_cols = min(3, len(present))
    n_rows = (len(present) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows),
                              squeeze=False)
    axes_flat = axes.flatten()

    for i, col in enumerate(present):
        ax = axes_flat[i]
        vals = bp3m_images[col].dropna().values
        ax.hist(vals, bins=20, color='steelblue', edgecolor='white', lw=0.5)
        ax.axvline(np.median(vals), color='darkorange', lw=1.5,
                   label=f'Median = {np.median(vals):.4g}')
        ax.set_xlabel(col)
        ax.set_ylabel('N images')
        ax.legend(fontsize=8)

    for j in range(len(present), len(axes_flat)):
        axes_flat[j].set_visible(False)

    fig.suptitle(f'{FIELD_NAME} — BP3M image transformation parameters', fontsize=13)
    plt.tight_layout()
    plt.savefig(DATA / FIELD_NAME / 'plots_04_transformations.png', dpi=150)
    plt.show()
else:
    print('BP3M results required for transformation parameter plots.')